Part 2: Board Game Recommendation

In [2]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Replace 'your_file.csv' with the actual path or name of your CSV file
df = pd.read_csv('boardgames_ranks.csv')
df.head()


,id,name,yearpublished,rank,bayesaverage,average,usersrated,is_expansion,abstracts_rank,cgs_rank,childrensgames_rank,familygames_rank,partygames_rank,strategygames_rank,thematic_rank,wargames_rank
0,224517,Brass: Birmingham,2018,1,8.39022,8.55904,60513,0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
1,342942,Ark Nova,2021,2,8.35515,8.53855,63479,0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
2,161936,Pandemic Legacy: Season 1,2015,3,8.34261,8.49972,57904,0,NaN,NaN,NaN,NaN,NaN,3.0,1.0,NaN
3,174430,Gloomhaven,2017,4,8.28935,8.53111,67835,0,NaN,NaN,NaN,NaN,NaN,5.0,2.0,NaN
4,397598,Dune: Imperium – Uprising,2023,5,8.26913,8.69975,20546,0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN


In [3]:
baseGames = df[df['is_expansion'] == 0]


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

# 1. Define our features (X) and target variable (y)
# We use only the category ranks as features. We MUST exclude the main 'rank' 
# column from X, otherwise we'd be giving the model the answer!
feature_columns = [col for col in baseGames.columns if col.endswith('_rank')]
X = baseGames[feature_columns].fillna(10000000)
y = baseGames['rank']

# 2. Set up GridSearchCV to find the optimal number of neighbors
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15]}
knn_reg = KNeighborsRegressor(metric='euclidean')

# Fit the grid search (this tests each n_neighbors value using cross-validation)
grid_search = GridSearchCV(knn_reg, param_grid, cv=5)
grid_search.fit(X, y)

# Extract the best performing model
best_knn = grid_search.best_estimator_
print(f"Best n_neighbors found by GridSearchCV: {grid_search.best_params_['n_neighbors']}")

# 3. Define the estimation function
def estimate_game_rank(game_name):
    # Find the index of the requested game
    if game_name not in baseGames['name'].values:
        return "Game not found in dataset."
        
    game_idx = baseGames[baseGames['name'] == game_name].index[0]
    
    # Extract the category rank features for this specific game
    game_features = X.loc[[game_idx]]
    
    # Predict the overall rank
    estimated_rank = best_knn.predict(game_features)[0]
    actual_rank = baseGames.loc[game_idx, 'rank']
    
    return {
        'Game': game_name,
        'Estimated Rank': int(round(estimated_rank)),
        'Actual Rank': actual_rank
    }

# Example usage with one of the games from your previous cell
print(estimate_game_rank('Clank!: A Deck-Building Adventure'))